# Customer Churn Prediction Pipeline

This notebook demonstrates a complete Machine Learning pipeline for predicting customer churn using the **Telco Customer Churn** dataset. We will compare a **Logistic Regression** model and a **Decision Tree Classifier** to determine which performs better at identifying customers likely to leave.

### 1. Import Libraries and Configuration
In this block, we import the necessary libraries for data manipulation, visualization, and machine learning. We also define our configuration constants such as feature names and model parameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Dataset URL
DATASET_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Feature Definitions
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", 
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", 
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", 
    "Contract", "PaperlessBilling", "PaymentMethod"
]
TARGET_COLUMN = "Churn"
DROP_COLUMNS = ["customerID"]

### 2. Data Loading
We load the dataset directly from the source URL and perform a quick inspection of the data structure and target distribution.

In [ ]:
df_raw = pd.read_csv(DATASET_URL)
print(f"Dataset Shape: {df_raw.shape}")
display(df_raw.head())

print("\nTarget Distribution:")
print(df_raw[TARGET_COLUMN].value_counts(normalize=True))

### 3. Data Cleaning
Here we clean the data by dropping unnecessary columns, converting 'TotalCharges' to numeric (handling empty strings as NaNs), and mapping our target labels ('Yes'/'No') to 1 and 0.

In [ ]:
df = df_raw.copy()
df.drop(columns=DROP_COLUMNS, inplace=True, errors="ignore")

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Encode Target
df[TARGET_COLUMN] = df[TARGET_COLUMN].map({"Yes": 1, "No": 0})

print(f"Cleaned Data Shape: {df.shape}")

### 4. Preprocessing Pipeline
We build a `ColumnTransformer` to automate preprocessing. 
- **Numeric features** are imputed with the median and then scaled.
- **Categorical features** are imputed with the most frequent value and then One-Hot encoded.

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

### 5. Model Training and Evaluation
We train two models: a **Logistic Regression** (linear classification) and a **Decision Tree**. For each model, we calculate key metrics like Accuracy, Precision, Recall, and F1-Score.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

df_results = pd.DataFrame(results)
display(df_results)

### 6. Model Comparison Visualization
This block generates a bar chart using Matplotlib to visually compare the performance metrics of both models. This helps in quickly identifying which model is more reliable for churn prediction.

In [ ]:
df_plot = df_results.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(data=df_plot, x="Metric", y="Score", hue="Model", palette="viridis")
plt.title("Model Comparison: Logistic Regression vs Decision Tree", fontsize=15)
plt.ylim(0, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()